In [ ]:
import pandas as pd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from pathlib import Path
from datetime import datetime

In [ ]:
#log to a file and print to terminal
# Open logging file
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
log_filename = f"Output/analyze_all_class_beacons_log_{timestamp}.txt"
log_file = open(log_filename, "a")# Open file in append mode
def log_and_print(message):
    print(message)  # Print to terminal is too noisy
    log_file.write(message + "\n")  # Write to file

log_and_print(f"Analysis started on {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# read in relevant data
all_class_beacons_df = pd.read_parquet('Data/all_class_beacons.parquet')
sample_aps_df = pd.read_parquet('Sample/sample_aps.parquet')
#print(sample_aps_df.head())

In [ ]:
print(all_class_beacons_df.sample(n=10))
#print(all_class_beacons_df.dtypes)
#print(all_class_beacons_df.head())

In [ ]:
# need to verify the calculations are correct
# create a column for total room station count
# This needs to be done before additional SSIDs are removed
# iterate through each location, find all APs in that location, and find the unique wlan.ta MAC addresses of all the APs
# then add the wlan.qbss.scount values for all the wlan.ta values for each beacon
# the result is the total number of stations in the room

# Step 1: Set up the mappings
# Map AP names in sample_aps_df to locations
hostname_to_location = sample_aps_df.set_index("Hostname")["Building-Room"].to_dict()
location_to_hostnames = sample_aps_df.groupby("Building-Room")["Hostname"].apply(list).to_dict()

print("step 1 done")
# Step 2: Prepare main DataFrame
df = all_class_beacons_df.sort_values("aruba_erm.time").reset_index(drop=True)
df["room_scount"] = 0

print("step 2 done")
# Step 3: Preprocess: create lookup per AP (wlan.vs.aruba.ap_name)
# Directory where your parquet files live
parquet_dir = Path("Data/sampled")

host_dfs = {}
host_time_arrays = {}
host_ta_arrays = {}
host_scount_arrays = {}

for hostname in tqdm(df["wlan.vs.aruba.ap_name"].unique(), desc="Preprocessing APs"):
    #parquet_path = parquet_dir / f"{hostname}_class_beacons.parquet"

    #if parquet_path.exists():
    #    try:
    #        host_df = pd.read_parquet(parquet_path)
    #        #print(f"Read {parquet_path}")
    #    except Exception as e:
    #        print(f"Failed to read {parquet_path}, falling back to in-memory filter: {e}")
    #        host_df = df[df["wlan.vs.aruba.ap_name"] == hostname].copy()
    #else:
    #    # Fallback if no .parquet file exists
    #    host_df = df[df["wlan.vs.aruba.ap_name"] == hostname].copy() 
    #    #print(f"No {parquet_path}, falling back to in-memory filter")   
    host_df = df[df["wlan.vs.aruba.ap_name"] == hostname].copy() # slower but more reliable

    host_df = host_df.sort_values("aruba_erm.time")
    times = host_df["aruba_erm.time"].astype("int64").values  # nanoseconds since epoch
    host_dfs[hostname] = host_df.reset_index(drop=True)
    host_time_arrays[hostname] = times
    host_ta_arrays[hostname] = host_df["wlan.ta"].values
    host_scount_arrays[hostname] = host_df["wlan.qbss.scount"].values

print("step 3 done")
# Step 4: Process each row
results = np.zeros(len(df), dtype=int)
five_seconds_ns = np.timedelta64(5, 's').astype('timedelta64[ns]').astype('int64')

for i in tqdm(range(len(df)), desc="Computing room_scount"):
    row = df.iloc[i]
    current_time = row["aruba_erm.time"]
    current_hostname = row["wlan.vs.aruba.ap_name"]
    current_ta = row["wlan.ta"]
    location = hostname_to_location.get(current_hostname)

    if not location:
        continue

    other_hostnames = [h for h in location_to_hostnames[location] if h != current_hostname]
    seen_ta = {current_ta}
    room_scount = row["wlan.qbss.scount"]
    current_time_ns = current_time.value

    ta_to_best_idx = {}

    for other_hostname in other_hostnames:
        if other_hostname not in host_time_arrays:
            continue

        times = host_time_arrays[other_hostname]         # int64 nanoseconds
        scounts = host_scount_arrays[other_hostname]
        tas = host_ta_arrays[other_hostname]

        # Get ALL rows from this AP within ±5 seconds of current time
        left_bound = current_time_ns - five_seconds_ns
        right_bound = current_time_ns + five_seconds_ns

        left_idx = np.searchsorted(times, left_bound, side='left')
        right_idx = np.searchsorted(times, right_bound, side='right')

        matching_indices = range(left_idx, right_idx)


        

        for j in matching_indices:
            ta = tas[j]
            time_diff = np.abs(times[j] - current_time_ns)
    
            if ta not in ta_to_best_idx or time_diff < np.abs(times[ta_to_best_idx[ta]] - current_time_ns):
                ta_to_best_idx[ta] = j

    for ta, j in ta_to_best_idx.items():
        if ta not in seen_ta:
            seen_ta.add(ta)
            room_scount += scounts[j]

    results[i] = int(room_scount)


# Step 5: Assign results back to the DataFrame
df["room_scount"] = results
all_class_beacons_df = df  # Optional: assign back to original

#remove this later:
all_class_beacons_df.to_parquet("Output/all_class_beacons_processed.parquet", engine="pyarrow")



In [ ]:
#reduce overrepresentation of areas that have additional SSIDs
print(all_class_beacons_df["wlan.ssid"].unique())
all_class_beacons_df = all_class_beacons_df[all_class_beacons_df["wlan.ssid"].isin(["eduroam", "BYU-WiFi"])] # update to use env file
print(all_class_beacons_df["wlan.ssid"].unique())


In [ ]:
print(len(all_class_beacons_df))

In [ ]:
print(all_class_beacons_df.dtypes)

In [ ]:
# metrics for all beacons during class sessions in a single dataframe

total_data_points = len(all_class_beacons_df)

# Count the number of data points where wlan.qbss.cu is greater than 191 (75%)
greater_than_191_count = (all_class_beacons_df["wlan.qbss.cu"] > 191).sum()

# Calculate the percentage of data points where wlan.qbss.cu is greater than 191
percentage_greater_than_191 = (greater_than_191_count / total_data_points) * 100

log_and_print(f"percent of beacons with very high utilization (> 75% cu): {percentage_greater_than_191:.4f} %")

# Count the number of data points where wlan.qbss.cu is greater than 127 (50%)
greater_than_127_count = (all_class_beacons_df["wlan.qbss.cu"] > 127).sum()

# Calculate the percentage of data points where wlan.qbss.cu is greater than 127
percentage_greater_than_127 = (greater_than_127_count / total_data_points) * 100

log_and_print(f"percent of beacons with high utilization (> 50% cu): {percentage_greater_than_127:.4f} %")

log_and_print(f"median number of associated devices per radio: {all_class_beacons_df['total_scount'].median()}")

log_and_print(f"average number of associated devices per radio: {all_class_beacons_df['total_scount'].mean():.2f}")

log_and_print(f"median channel utilization qbss: {all_class_beacons_df['wlan.qbss.cu'].median()}")

log_and_print(f"average channel utilization qbss: {all_class_beacons_df['wlan.qbss.cu'].mean():.2f}")

In [ ]:

# Create histogram for the 'wlan.qbss.cu' column
plt.hist(all_class_beacons_df["wlan.qbss.cu"], bins=range(256), edgecolor='black', align='left')

# Add labels and title
plt.xlabel('wlan.qbss.cu values')
plt.ylabel('Frequency')
plt.title('Histogram of wlan.qbss.cu Values (0-255)')

plot_file = "Output/all_class_beacons/Channel_use_histogram.png"
#save to a file
plt.savefig(plot_file)

# Show the plot
plt.show()




In [ ]:
# Set the number of bins to be the highest value of total_scount + 1
num_bins = max(all_class_beacons_df["total_scount"]) + 1

# Create histogram
plt.hist(all_class_beacons_df["total_scount"], bins=num_bins, edgecolor='black')

# Add labels and title
plt.xlabel('total_scount values')
plt.ylabel('Frequency')
plt.title('Histogram of total_scount Values')

plot_file = "Output/all_class_beacons/Scount_histogram.png"
#save to a file
plt.savefig(plot_file)

# Show the plot
plt.show()

In [ ]:
# Calculate the pearson correlation between wlan.qbss.cu and total_scount
correlation = all_class_beacons_df["total_scount"].corr(all_class_beacons_df["wlan.qbss.cu"])

# Display the result
log_and_print(f"The correlation between total_scount and wlan.qbss.cu is: {correlation}")


In [ ]:

# create a hexbin plot to visualize the correlation between station count and channel utilization

plt.hexbin(
    all_class_beacons_df["total_scount"],  # Now on x-axis
    all_class_beacons_df["wlan.qbss.cu"],  # Now on y-axis
    gridsize=50,
    cmap="Blues",
    norm=mcolors.LogNorm()  # Log scaling for better visibility
)

plt.xlabel("total_scount")  # X-axis: station count
plt.ylabel("wlan.qbss.cu")  # Y-axis: channel utilization
plt.title("Hexbin Plot: Channel Utilization vs. Station Count")

plt.colorbar(label="Log Count")

plot_file = "Output/all_class_beacons/Scount_channel_use_hexbin.png"
#save to a file
plt.savefig(plot_file)

plt.show()


In [ ]:
from scipy.stats import linregress

x = all_class_beacons_df["total_scount"]
y = all_class_beacons_df["wlan.qbss.cu"]

slope, intercept, r_value, p_value, std_err = linregress(x, y)

log_and_print(f"Slope: {slope:.3f}")
log_and_print(f"Intercept: {intercept:.3f}")
log_and_print(f"R-squared: {r_value**2:.3f}")
log_and_print(f"P-value: {p_value:.3g}")

log_file.close()

#255 = .442x + 21.618
#x = 528.01
# this model suggests that 528 clients would use about 100% of airtime
#127.5 = .442x + 21.618
#x = 239.4
#also suggests that 239 clients would use 50% of airtime

In [ ]:
#write out if needed
all_class_beacons_df.to_parquet("Output/all_class_beacons_processed.parquet", engine="pyarrow")